# Explore diagram rules with `suggest`

Rate each diagram with **▲ upvote**, **▼ downvote**, or **? unsure**. Both pictures show the same data. Once both are rated, the next pair appears.

The LLM proposes whole programs; Hypothesis checks generated trees and searches for useful comparisons; Spytial's solver constructs the contrasting layouts. You do not need to read Spytial code.

Run the cells through **Open the session**. Use a local Jupyter kernel with this checkout's dependencies, the `suggest-search` extra, Node, and an authenticated `codex` CLI. Keep the kernel running while using the session.


In [ ]:
from pathlib import Path
import os
import sys

# Find this checkout whether Jupyter starts in demos/, spytial-py/, or spytial-org/.
cwd = Path.cwd()
repo = next((p for p in (cwd, cwd.parent, cwd / "spytial-py")
             if (p / "spytial" / "suggest" / "_session.py").is_file()), None)
if repo is None:
    raise RuntimeError("Open this notebook from the spytial-py checkout or its demos folder.")
sys.path.insert(0, str(repo))

# Prefer the locally built renderer when available; otherwise use the pinned CDN.
bundle = repo.parent / "spytial-core" / "dist" / "browser" / "spytial-core-complete.global.js"
if bundle.is_file():
    os.environ.setdefault("SPYTIAL_BROWSER_BUNDLE", str(bundle))

from dataclasses import dataclass
from typing import Optional
from IPython.display import display
from spytial.suggest import suggest


In [ ]:
# An empty list lets the model propose starting rules.
requested_rules = [
    "Put children below their parents",
    "Align children",
]


In [ ]:
@dataclass
class Tree:
    value: int
    left: Optional["Tree"] = None
    right: Optional["Tree"] = None

examples = [
    Tree(8, Tree(3), Tree(10)),
    Tree(8, Tree(3, Tree(1), Tree(6)), Tree(10, None, Tree(14))),
]

from hypothesis import strategies as st

# A bounded family of valid trees; no cycles or shared children.
def trees(depth):
    children = st.none() if depth == 0 else st.one_of(st.none(), trees(depth - 1))
    return st.builds(Tree, value=st.integers(0, 20), left=children, right=children)

strategy = trees(2)


## Open the session

The view opens immediately while proposals and checks run in the background. Each vote is saved as you click it. Rerunning this cell redisplays the same session.

To add a rule later, use `session.revise("...")`. That keeps your previous judgments. To start over explicitly, close the session and delete the `session` variable first.


In [ ]:
if "session" not in globals():
    session = suggest(
        Tree,
        rules=requested_rules,
        examples=examples,
        strategy=strategy,
        max_examples=10,
    )
display(session)


## Inspect your progress (optional)

Run this after responding in the embedded view. This shows your own judgments without exposing Spytial code. No response is filled in for you.


In [ ]:
[answer["votes"] for answer in session.history]


## Revise or export later (optional)

A revision uses your earlier judgments and updates the same embedded view. Uncomment the first line to try one. `finish()` returns a draft without applying it; print it only if you want to inspect Spytial.


In [ ]:
# session.revise("All nodes at the same depth should share a row")
# draft = session.finish()
# print(draft.to_source())


## Close the session when finished

This stops the local server, so the embedded controls will stop working. Leave the line commented while exploring.

This prototype varies positioning while keeping hiding, labels, and styles fixed. Judgments are attached to the examples shown; they do not automatically become universal rules. “No further comparison found” means the available examples and search budget did not reveal another question, not that the proposals are equivalent or finished.


In [ ]:
# session.close()
